### 🦜🕸️ LangGraph 계층적 에이전트(Hierarchical Agent) 구조 설명

방금 실행한 코드는 **LangGraph**를 사용하여 **상위 그래프(Parent Graph)** 안에 **하위 에이전트(Sub-agent)** 를 노드로 포함시키는 계층적 구조를 보여줍니다.

#### 1. 구조 개요
이 시스템은 두 단계로 구성됩니다:
- **Supervisor(관리자)**: 사용자의 의도를 파악하거나 전처리를 담당합니다.
- **Math Expert(수학 전문가)**: 실제 계산 작업을 수행하는 하위 에이전트입니다.

#### 2. 핵심 구성 요소

**A. 하위 에이전트 (`math_agent`)**
- `create_agent`를 사용해 만든 독립적인 `Runnable` 객체입니다.
- `gpt-4o-mini` 모델과 수학 도구(`add`, `multiply`, `divide`)가 바인딩되어 있습니다.
- 입력된 메시지를 처리하고 필요시 도구를 호출하여 결과를 반환합니다.

**B. 부모 상태 (`ParentState`)**
- 기본 `messages` 외에 `user_category`라는 추가 필드를 가집니다.
- 이를 통해 그래프 간에 더 풍부한 문맥 정보를 공유할 수 있습니다.

**C. 노드 및 엣지 구성**
1. **Supervisor Node**:
   - `START`에서 가장 먼저 호출됩니다.
   - 사용자 입력을 분석해 `user_category`를 설정합니다 (예: "math_query").
2. **Math Expert Node**:
   - `math_agent` 자체가 하나의 노드로 추가되었습니다.
   - LangGraph에서는 다른 체인이나 에이전트(`Runnable`)를 그대로 노드로 사용할 수 있습니다.

#### 3. 실행 흐름
1. `START` → `supervisor`: 사용자 입력 분석 및 카테고리 태깅
2. `supervisor` → `math_expert`: 수학 에이전트에게 메시지 전달
3. `math_expert` (내부 루프): 모델 호출 ↔ 도구 실행 반복 (ReAct 패턴)
4. `math_expert` → `END`: 최종 답변 반환

이러한 구조는 복잡한 작업을 여러 전문 에이전트에게 분배하는 **Multi-Agent System**의 기초가 됩니다.

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv("/home/user/MultiAgents/.env")

# 모델 정의
model = init_chat_model(
    "gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # 핵심: OpenAI SDK의 base_url과 동일 개념
    temperature=0,    
)

In [ ]:
from typing import Annotated
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

# ===== 상태(State) 타입 정의 =====
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # 대화 메시지 목록을 상태로 저장합니다.


# ===== 도구(Tool) 정의 =====
@tool
def multiply(a: int, b: int) -> int:
    """두 숫자를 곱합니다."""
    return a * b  # 두 정수를 곱한 결과를 반환합니다.

@tool
def add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    return a + b  # 두 정수를 더한 결과를 반환합니다.

@tool
def divide(a: int, b: int) -> float:
    """두 숫자를 나눕니다."""
    if b == 0:
        return "오류: 0으로 나눌 수 없습니다."  # 0으로 나누는 경우 오류 메시지를 반환합니다.
    return a / b  # 두 수를 나눈 실수 결과를 반환합니다.


# ===== LLM + Tools 바인딩 =====
tools = [add, multiply, divide]  # 에이전트가 사용할 도구 목록을 정의합니다.
llm_with_tools = model.bind_tools(tools)  # 모델에 도구 호출 기능을 연결합니다.


# ===== Agent 노드 =====
def agent_node(state: AgentState):
    """LLM을 호출하는 노드"""
    response = llm_with_tools.invoke(state["messages"])  # 현재까지의 메시지를 기반으로 모델을 호출합니다.
    return {"messages": [response]}  # 모델 응답을 messages 형태로 반환합니다.


# ===== 그래프 빌드 =====
graph_builder = StateGraph(AgentState)  # AgentState를 사용하는 상태 그래프를 생성합니다.

graph_builder.add_node("agent", agent_node)  # LLM을 호출하는 agent 노드를 추가합니다.
graph_builder.add_node("tools", ToolNode(tools))  # 도구 실행을 담당하는 tools 노드를 추가합니다.

graph_builder.add_edge(START, "agent")  # 시작 지점에서 agent 노드로 이동하도록 연결합니다.
graph_builder.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "tools", "__end__": "__end__"}  # 도구 호출이 필요하면 tools로, 아니면 종료합니다.
)
graph_builder.add_edge("tools", "agent")  # 도구 실행 후 다시 agent로 돌아오도록 연결합니다.

graph = graph_builder.compile()  # 정의한 그래프를 실행 가능한 객체로 컴파일합니다.


# ===== 테스트 함수 =====
def run_agent(user_input: str):
    """일반 실행"""
    print(f"\n💬 사용자 입력: {user_input}")  # 사용자 입력을 출력합니다.
    print("=" * 60)  # 구분선을 출력합니다.
    result = graph.invoke({"messages": [HumanMessage(content=user_input)]})  # 사용자 입력을 그래프에 전달해 실행합니다.
    last_message = result["messages"][-1]  # 최종 응답 메시지를 가져옵니다.
    print(f"🤖 Agent 응답: {last_message.content}")  # 에이전트의 최종 응답을 출력합니다.
    print("=" * 60)  # 구분선을 출력합니다.


def stream_agent(user_input: str):
    """스트리밍 실행 (노드별 과정 확인)"""
    print(f"\n💬 사용자 입력: {user_input}")  # 사용자 입력을 출력합니다.
    print("=" * 60)  # 구분선을 출력합니다.
    for event in graph.stream(
        {"messages": [HumanMessage(content=user_input)]},  # 사용자 메시지를 초기 상태로 전달합니다.
        stream_mode="updates"  # 각 노드의 업데이트 과정을 순차적으로 스트리밍합니다.
    ):
        for node, value in event.items():  # 각 이벤트에서 노드 이름과 값을 순회합니다.
            if node != "__end__" and "messages" in value:  # 종료 노드가 아니고 messages가 있으면 처리합니다.
                last_msg = value["messages"][-1]  # 해당 노드의 마지막 메시지를 가져옵니다.
                if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:  # 도구 호출 정보가 있으면 출력합니다.
                    for tc in last_msg.tool_calls:
                        print(f"  🔧 [{node}] 도구 호출: {tc['name']}({tc['args']})")  # 어떤 도구가 어떤 인자로 호출됐는지 보여줍니다.
                elif hasattr(last_msg, 'content') and last_msg.content:  # 일반 텍스트 응답이 있으면 출력합니다.
                    msg_type = last_msg.__class__.__name__  # 메시지 클래스 이름을 확인합니다.
                    print(f"  📨 [{node}] {msg_type}: {last_msg.content}")  # 노드별 메시지 내용을 출력합니다.
    print("=" * 60)  # 구분선을 출력합니다.

### 프롬프트 기반 동작
* 프롬프트가 tool call이 필요없는 요청인 경우 모델 자체 답변으로 진행합니다.

In [ ]:
# ===== 테스트 실행 =====
if __name__ == "__main__":
    # 테스트 1: 단일 도구 호출
    run_agent("25와 4를 곱해줘")

    # 테스트 2: 다중 도구 호출 (체이닝)
    run_agent("100을 5로 나누고 결과에 2를 더해줘")

In [ ]:
# ===== 테스트 실행 =====
if __name__ == "__main__":
    # 테스트 3: 스트리밍으로 과정 확인
    stream_agent("10, 20, 30의 합은?")

    # 테스트 4: 도구 불필요한 일반 질문
    stream_agent("안녕하세요!")

# 5. LangGraph 그래프 직접 구현 연습
## 분기(Branching), 루프(Loop) 패턴

## 📚 목차

1. **예제 1: 조건부 분기 (Conditional Branching)** — LLM 기반 라우팅
2. **예제 2: 루프와 자기 수정 (Loop with Self-Correction)** — 재시도 패턴

**참고 문서:**
- [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [LangGraph Best Practices](https://www.swarnendu.de/blog/langgraph-best-practices/)
- [Command 공식 블로그](https://blog.langchain.com/command-a-new-tool-for-multi-agent-architectures-in-langgraph/)

---
## 예제 1: 조건부 분기 (Conditional Branching)
---

### 📌 개요
사용자 입력을 LLM이 분류(math / general / translate)한 뒤,
각 전문 노드로 분기하는 **라우팅 패턴**입니다.

### 🔀 그래프 구조
```
START → classifier → (조건부 분기)
                      ├─→ math_handler → END
                      ├─→ translate_handler → END
                      └─→ general_handler → END
```

### 💡 학습 포인트
- `add_conditional_edges`로 동적 분기 구현
- LLM을 라우터로 활용하는 패턴
- 라우팅 함수의 반환값과 노드 매핑

In [ ]:
from typing import Annotated, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

# ===== 상태(State) 정의 =====
class RouterState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # 대화 메시지 목록을 상태로 저장합니다.
    category: str  # 라우팅 분류 결과를 저장합니다. ("math", "translate", "general")


# ===== LLM 초기화 =====
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 사용할 OpenAI 호환 채팅 모델 이름을 지정합니다.
    api_key=os.environ["PROXY_TOKEN"],  # 프록시 인증에 사용할 API 토큰을 환경변수에서 불러옵니다.
    base_url=os.environ["PROXY_URL"],  # OpenAI 기본 엔드포인트 대신 프록시 서버 주소를 사용합니다.
    temperature=0,  # 출력의 랜덤성을 0으로 설정해 일관된 응답을 유도합니다.
)

In [ ]:
# ===== 노드(Node) 정의 =====
def classifier_node(state: RouterState):
    """LLM이 사용자 입력을 분류합니다."""
    last_message = state["messages"][-1].content  # 가장 최근 사용자 메시지의 텍스트만 추출합니다.

    response = llm.invoke([
        SystemMessage(content=(
            "사용자 입력을 다음 카테고리 중 하나로 분류하세요. "  # 분류 작업의 규칙을 시스템 메시지로 전달합니다.
            "반드시 카테고리 이름만 출력하세요:\n"  # 출력 형식을 카테고리 이름 하나로 제한합니다.
            "- math: 수학 계산 관련\n"  # math 카테고리의 의미를 설명합니다.
            "- translate: 번역 관련\n"  # translate 카테고리의 의미를 설명합니다.
            "- general: 그 외 일반 질문"  # general 카테고리의 의미를 설명합니다.
        )),
        HumanMessage(content=last_message)  # 실제 사용자 입력을 분류 대상으로 전달합니다.
    ])

    category = response.content.strip().lower()  # 모델 응답에서 공백을 제거하고 소문자로 정규화합니다.
    # 예상치 못한 응답 방어
    if category not in ("math", "translate", "general"):
        category = "general"  # 지정된 카테고리가 아니면 기본값으로 general을 사용합니다.

    print(f"  🏷️ 분류 결과: {category}")  # 현재 분류 결과를 콘솔에 출력합니다.
    return {"category": category}  # 이후 라우팅에 사용할 category 값을 상태로 반환합니다.


def math_handler(state: RouterState):
    """수학 관련 질문을 처리합니다."""
    response = llm.invoke([
        SystemMessage(content="당신은 수학 전문가입니다. 정확한 계산 결과를 제공하세요."),  # 수학 전용 응답 역할을 시스템 메시지로 지정합니다.
        state["messages"][-1]  # 마지막 사용자 메시지를 그대로 전달해 수학 질문에 답하게 합니다.
    ])
    return {"messages": [response]}  # 수학 처리 결과를 messages 형식으로 반환합니다.


def translate_handler(state: RouterState):
    """번역 요청을 처리합니다."""
    response = llm.invoke([
        SystemMessage(content="당신은 전문 번역가입니다. 정확하고 자연스러운 번역을 제공하세요."),  # 번역 전용 응답 역할을 시스템 메시지로 지정합니다.
        state["messages"][-1]  # 마지막 사용자 메시지를 그대로 전달해 번역하게 합니다.
    ])
    return {"messages": [response]}  # 번역 결과를 messages 형식으로 반환합니다.


def general_handler(state: RouterState):
    """일반 질문을 처리합니다."""
    response = llm.invoke([
        SystemMessage(content="당신은 친절한 AI 어시스턴트입니다."),  # 일반 질문에 대한 기본 응답 역할을 지정합니다.
        state["messages"][-1]  # 마지막 사용자 메시지를 전달해 일반 답변을 생성합니다.
    ])
    return {"messages": [response]}  # 일반 질문 처리 결과를 messages 형식으로 반환합니다.

In [ ]:
# ===== 라우팅 함수 =====
def route_by_category(state: RouterState) -> Literal["math_handler", "translate_handler", "general_handler"]:
    """분류 결과에 따라 라우팅합니다."""
    category = state["category"]  # classifier_node에서 저장한 분류 결과를 가져옵니다.
    if category == "math":
        return "math_handler"  # 수학 관련 질문이면 math_handler 노드로 이동합니다.
    elif category == "translate":
        return "translate_handler"  # 번역 관련 질문이면 translate_handler 노드로 이동합니다.
    else:
        return "general_handler"  # 그 외의 경우에는 general_handler 노드로 이동합니다.

In [ ]:
# ===== 그래프 빌드 =====
graph_builder = StateGraph(RouterState)

# 노드 추가
graph_builder.add_node("classifier", #TODO)
graph_builder.add_node("math_handler", #TODO)
graph_builder.add_node("translate_handler", #TODO)
graph_builder.add_node("general_handler", #TODO)

# 엣지 구성
graph_builder.add_edge(START, "classifier")

# 🔑 핵심: 조건부 엣지 — classifier 이후 category에 따라 분기
graph_builder.add_conditional_edges(
    "classifier",                    # 출발 노드
    route_by_category,               # 라우팅 함수
    {                                # 반환값 → 노드 매핑
        "math_handler": "#TODO",
        "translate_handler": "#TODO",
        "general_handler": "#TODO",
    }
)

# 각 핸들러 → END
graph_builder.add_edge("math_handler", END)
graph_builder.add_edge("translate_handler", END)
graph_builder.add_edge("general_handler", END)

router_graph = graph_builder.compile()

### 🧪 예제 1 테스트

In [ ]:
print("=" * 60)
print("🔀 예제 1: 조건부 분기 (Conditional Branching)")
print("=" * 60)

test_inputs = [
    "345 곱하기 27은?",
    "Hello world를 한국어로 번역해줘",
    "오늘 날씨가 좋다. 뭐 하면 좋을까?",
]

for user_input in test_inputs:
    print(f"\n💬 입력: {user_input}")
    result = router_graph.invoke({
        "messages": [HumanMessage(content=user_input)],
        "category": ""
    })
    print(f"🤖 응답: {result['messages'][-1].content}")
    print("-" * 40)

---
## 예제 2: 루프와 자기 수정 (Loop with Self-Correction)
---

### 📌 개요
LLM이 생성한 답변을 **검증(validate)** 하고,
품질이 부족하면 **피드백과 함께 재생성**하는 루프 패턴입니다.

### 🔄 그래프 구조
```
START → generator → validator → (조건부 분기)
                                  ├─→ generator (재시도, 최대 3회)
                                  └─→ END (통과)
```

### 💡 학습 포인트
- 루프를 통한 자기 수정(self-correction) 패턴
- `retry_count`로 무한 루프 방지 (max_retries)
- 검증 노드에서의 조건부 분기

In [ ]:
class WriterState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # 대화 메시지 목록을 상태에 저장합니다.
    topic: str               # 작성할 글의 주제를 저장합니다.
    draft: str               # 현재 생성된 초안 내용을 저장합니다.
    feedback: str            # 검증 단계에서 받은 피드백 내용을 저장합니다.
    retry_count: int         # 초안 수정 및 재생성 시도 횟수를 저장합니다.
    is_approved: bool        # 현재 초안이 승인되었는지 여부를 저장합니다.

MAX_RETRIES = 3  # 최대 재시도 횟수를 3회로 설정합니다.

In [ ]:
def generator_node(state: WriterState):
    """주제에 대한 글을 작성합니다. 피드백이 있으면 반영합니다."""
    topic = state["topic"]  # 현재 글쓰기 주제를 상태에서 가져옵니다.
    feedback = state.get("feedback", "")  # 이전 검증 단계에서 받은 피드백을 가져오고, 없으면 빈 문자열을 사용합니다.
    retry_count = state.get("retry_count", 0)  # 현재까지 수정 시도 횟수를 가져오고, 없으면 0으로 시작합니다.

    if feedback:
        prompt = (
            f"주제: {topic}\n\n"  # 글의 주제를 프롬프트에 포함합니다.
            f"이전 초안에 대한 피드백:\n{feedback}\n\n"  # 이전 초안에 대한 피드백을 함께 전달합니다.
            f"피드백을 반영하여 개선된 글을 작성하세요. "  # 피드백을 반영한 개선 초안 작성을 요청합니다.
            f"반드시 3문장 이상으로 구체적인 예시를 포함하세요."  # 최소 문장 수와 예시 포함 조건을 명시합니다.
        )
        print(f"  ✏️ [{retry_count + 1}차 수정] 피드백 반영 중...")  # 수정 단계임을 콘솔에 출력합니다.
    else:
        prompt = (
            f"주제: {topic}\n\n"  # 초안 작성 주제를 프롬프트에 포함합니다.
            f"이 주제에 대해 3문장 이상의 구체적인 글을 작성하세요. "  # 최소 3문장 이상의 구체적인 글 작성을 요청합니다.
            f"예시나 데이터를 포함하면 좋습니다."  # 예시나 데이터를 넣도록 유도합니다.
        )
        print(f"  ✏️ [초안 작성 중...]")  # 첫 초안 작성 단계임을 콘솔에 출력합니다.

    response = llm.invoke([HumanMessage(content=prompt)])  # 작성 프롬프트를 LLM에 전달하여 초안을 생성합니다.
    draft = response.content  # 모델이 생성한 글 초안을 추출합니다.

    print(f"  📝 초안: {draft[:80]}...")  # 생성된 초안의 앞부분만 미리보기로 출력합니다.
    return {
        "draft": draft,  # 생성된 초안을 상태로 반환합니다.
        "retry_count": retry_count + 1  # 시도 횟수를 1 증가시켜 다음 단계로 전달합니다.
    }


def validator_node(state: WriterState):
    """초안의 품질을 검증합니다."""
    draft = state["draft"]  # 검증할 초안을 상태에서 가져옵니다.
    print(f"  🔍 검증 중... (시도 {state['retry_count']}/{MAX_RETRIES})")  # 현재 검증 시도 횟수를 출력합니다.

    response = llm.invoke([
        SystemMessage(content=(
            "당신은 엄격한 편집자입니다. 다음 글을 평가하세요.\n"  # 모델에게 엄격한 편집자 역할을 부여합니다.
            "평가 기준:\n"  # 아래 기준으로 평가하도록 안내합니다.
            "1. 3문장 이상인가?\n"  # 문장 수 기준을 확인합니다.
            "2. 구체적인 예시나 데이터가 있는가?\n"  # 구체성 기준을 확인합니다.
            "3. 논리적으로 일관성이 있는가?\n\n"  # 논리적 일관성 기준을 확인합니다.
            "모든 기준을 충족하면 첫 줄에 'APPROVED'를 쓰세요.\n"  # 모든 기준 충족 시 승인 형식을 지정합니다.
            "부족하면 첫 줄에 'REJECTED'를 쓰고, 구체적인 개선 피드백을 제공하세요."  # 부족할 경우 반려와 피드백 형식을 지정합니다.
        )),
        HumanMessage(content=f"다음 글을 평가하세요:\n\n{draft}")  # 실제 초안을 평가 대상으로 전달합니다.
    ])

    result = response.content  # 검증 결과 전체 텍스트를 저장합니다.
    is_approved = result.strip().upper().startswith("APPROVED")  # 첫 줄이 APPROVED로 시작하는지 확인해 승인 여부를 판단합니다.

    if is_approved:
        print(f"  ✅ 승인됨!")  # 승인된 경우 콘솔에 성공 메시지를 출력합니다.
    else:
        print(f"  ❌ 반려 — 피드백 전달")  # 반려된 경우 피드백 전달 메시지를 출력합니다.

    return {
        "feedback": result,  # 검증 결과 또는 피드백을 상태로 반환합니다.
        "is_approved": is_approved  # 승인 여부를 상태로 반환합니다.
    }

In [ ]:
# 🔑 핵심: 루프 조건 함수
def should_retry(state: WriterState) -> Literal["generator", "__end__"]:
    """승인 여부와 재시도 횟수를 확인하여 루프를 제어합니다."""
    if state["is_approved"]:
        return "__end__"  # 검증을 통과했다면 더 이상 수정하지 않고 그래프를 종료합니다.
    if state["retry_count"] >= MAX_RETRIES:
        print(f"  ⚠️ 최대 재시도 횟수({MAX_RETRIES}) 도달 — 강제 종료")  # 재시도 한도에 도달했음을 출력합니다.
        return "__end__"  # 최대 횟수에 도달하면 승인 여부와 관계없이 종료합니다.
    return "generator"  # 아직 승인되지 않았고 재시도 가능하면 generator 노드로 돌아가 다시 글을 생성합니다.

In [ ]:
# ===== 그래프 빌드 =====
writer_builder = StateGraph(WriterState)

writer_builder.add_node("generator", #TODO)
writer_builder.add_node("validator", #TODO)

writer_builder.add_edge(START, "generator")
writer_builder.add_edge("#TODO", "validator")

# 🔑 핵심: validator → generator (루프) 또는 END
writer_builder.add_conditional_edges(
    "validator",
    should_retry,
    {
        "generator": "generator",  # 재시도 루프
        "__end__": "__end__",      # 종료
    }
)

writer_graph = writer_builder.compile()

### 🧪 예제 2 테스트

In [ ]:
print("=" * 60)
print("🔄 예제 2: 루프와 자기 수정 (Self-Correction Loop)")
print("=" * 60)

result = writer_graph.invoke({
    "messages": [],
    "topic": "AI가 교육 분야에 미치는 영향",
    "draft": "",
    "feedback": "",
    "retry_count": 0,
    "is_approved": False,
})

print(f"\n📄 최종 결과:")
print(f"  승인 여부: {'✅ 승인' if result['is_approved'] else '⚠️ 미승인 (최대 재시도 초과)'}")
print(f"  총 시도 횟수: {result['retry_count']}")
print(f"  최종 초안:\n{result['draft']}")